In [1]:
import os
import cv2
import csv
import mediapipe as mp
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from collections import defaultdict, Counter, deque
from scipy.spatial.distance import cdist
from collections import Counter
import joblib

c:\Users\nicol\.venv11\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [7]:
#extract keypoints from images and save to CSV

def extract_keypoints(dataset_path, csv_file):
    file_list = {}

    for cls in os.listdir(dataset_path):
        cls_path = os.path.join(dataset_path, cls)
        if os.path.isdir(cls_path):
            images = [os.path.join(cls_path, f) for f in os.listdir(cls_path)]
            file_list[cls] = images

    landmarks = ['class']
    for val in range(1, 22):
        landmarks += ['x{}'.format(val), 'y{}'.format(val)]  # Header CSV: class + 21 keypoints (x,y)

    with open(csv_file, mode='w', newline='') as f: 
        csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        csv_writer.writerow(landmarks)

    mp_hands = mp.solutions.hands
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands: #if mediaPiepe is not too confident, skip the image

        for cls, files in file_list.items(): # iterate through each class and its files
            print(f"Processing class: {cls}") 

            for file in files:
                image = cv2.imread(file)
                if image is None:
                    continue
                image = cv2.flip(image, 1)
                results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)) #process image to find hand landmarks

                try:
                    for hand_landmark in results.multi_hand_landmarks: #if hand landmarks are found
                        keypoints = list(np.array([[lm.x, lm.y] for lm in hand_landmark.landmark]).flatten()) # extract x,y coordinates of each landmark
                        row = [cls] + keypoints
                        with open(csv_file, mode='a', newline='') as f:
                            csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                            csv_writer.writerow(row)
                except:
                    pass

In [ ]:
train_path = "asl_alphabet_train"

#extract train keypoints and save to CSV

extract_keypoints(train_path, "hand_dataset_train.csv")


Processing class: A
Processing class: B
Processing class: C
Processing class: D
Processing class: E
Processing class: F
Processing class: G
Processing class: H
Processing class: I
Processing class: J
Processing class: K
Processing class: L
Processing class: M
Processing class: N
Processing class: O
Processing class: P
Processing class: Q
Processing class: R
Processing class: S
Processing class: T
Processing class: U
Processing class: V
Processing class: W
Processing class: X
Processing class: Y
Processing class: Z


In [ ]:
test_path = "asl_alphabet_test"

#extract test keypoints and save to CSV

extract_keypoints(test_path, "hand_dataset_test.csv")

Processing class: A
Processing class: B
Processing class: C
Processing class: D
Processing class: E
Processing class: F
Processing class: G
Processing class: H
Processing class: I
Processing class: J
Processing class: K
Processing class: L
Processing class: M
Processing class: N
Processing class: O
Processing class: P
Processing class: Q
Processing class: R
Processing class: S
Processing class: T
Processing class: U
Processing class: V
Processing class: W
Processing class: X
Processing class: Y
Processing class: Z


In [2]:
#Load train and test datasets with keypoint's coordinates

train_dataset = pd.read_csv("hand_dataset_train.csv")

X = train_dataset.iloc[:, 1:].values
Y = train_dataset.iloc[:, 0].values

# Split in train and validation sets

X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

test_dataset = pd.read_csv("hand_dataset_test.csv")

X_test = test_dataset.iloc[:, 1:].values
y_test = test_dataset.iloc[:, 0].values


In [3]:
#standardize keypoints coordinates

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Observations and features train:", X_train.shape)
print("Observations and features validation:", X_val.shape)
print("Observations and features test:", X_test.shape)

Observations and features train: (21576, 42)
Observations and features validation: (5394, 42)
Observations and features test: (2952, 42)


In [4]:
#Validation set is used to find the best k

k_values = list(range(1, 21))  #try k from 1 to 20
best_k = 1
best_val_acc = 0

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)             # train with a specific k on the training set
    y_val_pred = knn.predict(X_val)       # prediction on the validation set
    val_acc = accuracy_score(y_val, y_val_pred)
    
    print(f"k={k}, Validation Accuracy={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k

print(f"\nBest k: {best_k} with validation accuracy={best_val_acc:.4f}")

k=1, Validation Accuracy=0.9359
k=2, Validation Accuracy=0.9164
k=3, Validation Accuracy=0.9166
k=4, Validation Accuracy=0.9067
k=5, Validation Accuracy=0.9030
k=6, Validation Accuracy=0.8978
k=7, Validation Accuracy=0.8893
k=8, Validation Accuracy=0.8891
k=9, Validation Accuracy=0.8860
k=10, Validation Accuracy=0.8817
k=11, Validation Accuracy=0.8804
k=12, Validation Accuracy=0.8758
k=13, Validation Accuracy=0.8754
k=14, Validation Accuracy=0.8684
k=15, Validation Accuracy=0.8648
k=16, Validation Accuracy=0.8634
k=17, Validation Accuracy=0.8591
k=18, Validation Accuracy=0.8576
k=19, Validation Accuracy=0.8552
k=20, Validation Accuracy=0.8519

Best k: 1 with validation accuracy=0.9359


In [5]:
#Re-train on training + validation set with the best k

X_train_full = np.vstack((X_train, X_val))
y_train_full = np.concatenate((y_train, y_val))

final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_full, y_train_full)

# Evaluate on the test set

y_test_pred = final_knn.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Test Accuracy con k={best_k}: {test_acc:.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test, y_test_pred))

Test Accuracy con k=1: 0.6457

Classification report (test set):
              precision    recall  f1-score   support

           A       0.48      0.92      0.63       115
           B       0.99      1.00      1.00       115
           C       0.77      1.00      0.87       104
           D       0.91      0.25      0.39       115
           E       0.89      1.00      0.94       115
           F       1.00      1.00      1.00       115
           G       0.00      0.00      0.00       115
           H       0.63      1.00      0.77       115
           I       0.58      1.00      0.73       115
           J       1.00      0.31      0.48       115
           K       0.86      1.00      0.93       115
           L       0.77      0.98      0.86       115
           M       0.00      0.00      0.00       115
           N       0.98      0.40      0.57       114
           O       0.98      0.64      0.78        89
           P       0.29      0.80      0.43       115
           Q    

c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [6]:
# Save the scaler and the final model

joblib.dump(scaler, "scaler.pkl")
joblib.dump(final_knn, "knn_model.pkl")

['knn_model.pkl']

In [7]:
# Conformal prediction adapted to k-NN classifier

# Split test into calibration and final test

X_calib, X_final, y_calib, y_final = train_test_split(X_test, y_test, test_size=0.8, stratify=y_test, random_state=42)

# Function for nonconformity score

labels = np.unique(y_train)
def nonconformity_score(x, y_label, X_train, y_train):
    mask_same_label = (y_train == y_label) 
    mask_other_label = (y_train != y_label) 

    dist_same = cdist([x], X_train[mask_same_label]) #distance to points with the same label
    dist_diff = cdist([x], X_train[mask_other_label]) #distance to points with different labels

    min_same = np.min(dist_same) if dist_same.size > 0 else np.inf 
    min_diff = np.min(dist_diff) if dist_diff.size > 0 else np.inf

    return min_same / min_diff if min_diff > 0 else np.inf # nonconformity score

# Compute non conformity scores for calibration set
calibration_scores = [nonconformity_score(x, y, X_train, y_train) for x, y in zip(X_calib, y_calib)]
calibration_scores = np.array(calibration_scores)

# Compute q_hat based on alpha
alpha = 0.3
q_hat = np.quantile(calibration_scores, 1 - alpha, interpolation='higher')

# Compute prediction sets for test set
prediction_sets = []
for x in X_final:
    preds = [label for label in labels if nonconformity_score(x, label, X_calib, y_calib) <= q_hat] #x is added if its nonconformity score is less than or equal to q_hat
    prediction_sets.append(preds)

# Show prediction sets for examples with cardinality > 1
for i, (x, preds, y_true) in enumerate(zip(X_final[:-1], prediction_sets[:-1], y_final[:-1])):
    if len(preds) > 1:
        covered = y_true in preds
        print(f"Test sample {i} - True label: {y_true} - Prediction set: {preds} - Covered: {covered}")

# Compute overall coverage
coverages = [y_true in preds for y_true, preds in zip(y_final, prediction_sets)]
coverage = np.mean(coverages)
print(f"Overall coverage: {coverage:.3f}")

# Average cardinality
avg_cardinality = np.mean([len(p) for p in prediction_sets])
print("Average cardinality of prediction sets:", avg_cardinality)


Test sample 66 - True label: S - Prediction set: ['M', 'S'] - Covered: True
Test sample 98 - True label: P - Prediction set: ['G', 'P'] - Covered: True
Test sample 118 - True label: G - Prediction set: ['G', 'Q'] - Covered: True
Test sample 165 - True label: R - Prediction set: ['R', 'U'] - Covered: True
Test sample 174 - True label: R - Prediction set: ['R', 'U'] - Covered: True
Test sample 329 - True label: E - Prediction set: ['E', 'S'] - Covered: True
Test sample 375 - True label: Q - Prediction set: ['G', 'Q'] - Covered: True
Test sample 480 - True label: G - Prediction set: ['G', 'Q'] - Covered: True
Test sample 539 - True label: T - Prediction set: ['A', 'N', 'T'] - Covered: True
Test sample 544 - True label: E - Prediction set: ['E', 'S'] - Covered: True
Test sample 582 - True label: Q - Prediction set: ['G', 'Q'] - Covered: True
Test sample 605 - True label: T - Prediction set: ['A', 'T'] - Covered: True
Test sample 703 - True label: T - Prediction set: ['A', 'T'] - Covered: T